In [6]:

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier


train = pd.read_csv("H:/codesoft/task2 data/fraudTrain.csv", nrows=20000)
test = pd.read_csv("H:/codesoft/task2 data/fraudTest.csv", nrows=5000)



train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()



drop_cols = [
    'trans_num', 'first', 'last', 'street',
    'city', 'state', 'zip', 'dob'
]

train = train.drop(columns=drop_cols, errors='ignore')
test = test.drop(columns=drop_cols, errors='ignore')


train['trans_date_trans_time'] = pd.to_datetime(train['trans_date_trans_time'], errors='coerce')
test['trans_date_trans_time'] = pd.to_datetime(test['trans_date_trans_time'], errors='coerce')

for df in [train, test]:
    df['hour'] = df['trans_date_trans_time'].dt.hour
    df['day'] = df['trans_date_trans_time'].dt.day
    df['month'] = df['trans_date_trans_time'].dt.month

train = train.drop(columns=['trans_date_trans_time'])
test = test.drop(columns=['trans_date_trans_time'])



categorical_cols = ['merchant', 'category', 'gender', 'job']



for col in categorical_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = test[col].map(lambda s: s if s in le.classes_ else 'unknown')
    
    le.classes_ = np.append(le.classes_, 'unknown')
    test[col] = le.transform(test[col])


X_train = train.drop(columns=['is_fraud'])
y_train = train['is_fraud']

X_test = test.drop(columns=['is_fraud'])
y_test = test['is_fraud']



model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',   # IMPORTANT for fraud detection
    random_state=42
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)



print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9958

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4982
           1       0.00      0.00      0.00        18

    accuracy                           1.00      5000
   macro avg       0.50      0.50      0.50      5000
weighted avg       0.99      1.00      0.99      5000

